In [16]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

pd.set_option("display.float_format", "{:.4f}".format)


In [17]:
df = pd.read_csv("../data/raw_drinking_water_potability.csv")

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3276 entries, 0 to 3275
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ph               2785 non-null   float64
 1   Hardness         3276 non-null   float64
 2   Solids           3276 non-null   float64
 3   Chloramines      3276 non-null   float64
 4   Sulfate          2495 non-null   float64
 5   Conductivity     3276 non-null   float64
 6   Organic_carbon   3276 non-null   float64
 7   Trihalomethanes  3114 non-null   float64
 8   Turbidity        3276 non-null   float64
 9   Potability       3276 non-null   int64  
dtypes: float64(9), int64(1)
memory usage: 256.1 KB


In [19]:
X = df.drop(columns="Potability")
y = df["Potability"]

y.value_counts(normalize=True)


Potability
0   0.6099
1   0.3901
Name: proportion, dtype: float64

In [20]:
numeric_features = X.columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features)
    ]
)


In [21]:
# Train-test split antes de imputar (evitar data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [22]:
# Fit solo con train
X_train_clean = preprocessor.fit_transform(X_train)
X_test_clean = preprocessor.transform(X_test)


In [23]:
# Volver a DataFrame para inspección
X_train_clean = pd.DataFrame(
    X_train_clean,
    columns=numeric_features,
    index=X_train.index
)

X_test_clean = pd.DataFrame(
    X_test_clean,
    columns=numeric_features,
    index=X_test.index
)


In [24]:
X_train_clean.isna().sum()

ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
dtype: int64

In [25]:
# Unir X limpio con y
df_train_clean = X_train_clean.copy()
df_train_clean["Potability"] = y_train

df_test_clean = X_test_clean.copy()
df_test_clean["Potability"] = y_test


In [26]:
df_train_clean.to_csv("new_water_potability_train_clean.csv", index=False)
df_test_clean.to_csv("new_water_potability_test_clean.csv", index=False)